# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

## 2. Datos

In [57]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [58]:
# Tu código aquí
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   Product           912 non-null    str    
 3   TypeName          912 non-null    str    
 4   Inches            912 non-null    float64
 5   ScreenResolution  912 non-null    str    
 6   Cpu               912 non-null    str    
 7   Ram               912 non-null    str    
 8   Memory            912 non-null    str    
 9   Gpu               912 non-null    str    
 10  OpSys             912 non-null    str    
 11  Weight            912 non-null    str    
 12  Price_in_euros    912 non-null    float64
dtypes: float64(2), int64(1), str(10)
memory usage: 92.8 KB


In [4]:
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.describe()

,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


Las columnas parecen tener un contenido complejo

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

Vamos a depurar las columnas de Ram y Weight para eliminar las unidades

In [59]:
df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)

Vamos a desglosar la columna ScreenResolution

In [60]:
df["ScreenResolution"].value_counts()

ScreenResolution
Full HD 1920x1080                                349
1366x768                                         211
IPS Panel Full HD 1920x1080                      163
IPS Panel Full HD / Touchscreen 1920x1080         32
Full HD / Touchscreen 1920x1080                   30
1600x900                                          14
Touchscreen 1366x768                              11
Quad HD+ / Touchscreen 3200x1800                  11
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     10
4K Ultra HD / Touchscreen 3840x2160                7
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
Touchscreen 2560x1440                              6
IPS Panel 4K Ultra HD 3840x2160                    5
IPS Panel Retina Display 2560x1600                 5
Touchscreen 2256x1504                              5
1440x900                                           4
IPS Panel 1366x768                                 4
IPS Panel Retina Display 2304x1440                 4
IPS Panel Touchscreen 2560x14

In [61]:
df_copy = df.copy()
# Touchscreen: 1 si el string contiene "Touchscreen", 0 si no
df_copy['Touchscreen'] = df_copy['ScreenResolution'].str.contains('Touchscreen').astype(int)

In [62]:
# IPS Panel: 1 si el string contiene "IPS Panel", 0 si no
df_copy['IPS_Panel'] = df_copy['ScreenResolution'].str.contains('IPS Panel').astype(int)

In [63]:
# Retina Display: 1 si el string contiene "Retina Display", 0 si no
df_copy['Retina_Display'] = df_copy['ScreenResolution'].str.contains('Retina Display').astype(int)

In [64]:
# Resolución (ancho x alto): extraer los dos números con regex y eliminar columna original
resolucion = df_copy['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
df_copy['Res_Width'] = resolucion[0].astype(int)
df_copy['Res_Height'] = resolucion[1].astype(int)
df_copy = df_copy.drop(columns=['ScreenResolution'])

In [65]:
# Sanity check después de la reorganización de la columna:
print(df_copy[['Touchscreen', 'IPS_Panel', 'Retina_Display', 'Res_Width', 'Res_Height']].head(10))

   Touchscreen  IPS_Panel  Retina_Display  Res_Width  Res_Height
0            0          0               0       1920        1080
1            0          0               0       1920        1080
2            0          0               0       1920        1080
3            0          0               0       1440         900
4            0          0               0       1920        1080
5            0          0               0       1366         768
6            0          1               0       1920        1080
7            0          0               0       1920        1080
8            0          0               0       1920        1080
9            0          1               0       3840        2160


Vamos ahora a depurar la columna 'CPU'

In [66]:
pd.set_option('display.max_rows', None)
print(df_copy['Cpu'].value_counts())
pd.set_option('display.max_rows', 60)

Cpu
Intel Core i5 7200U 2.5GHz              124
Intel Core i7 7700HQ 2.8GHz             105
Intel Core i7 7500U 2.7GHz               97
Intel Core i5 8250U 1.6GHz               52
Intel Core i7 8550U 1.8GHz               47
Intel Core i3 6006U 2GHz                 45
Intel Core i5 6200U 2.3GHz               44
Intel Core i7 6500U 2.5GHz               40
Intel Core i7 6700HQ 2.6GHz              30
Intel Celeron Dual Core N3060 1.6GHz     25
Intel Core i3 7100U 2.4GHz               22
Intel Celeron Dual Core N3350 1.1GHz     22
Intel Core i5 7300HQ 2.5GHz              21
Intel Core i3 6006U 2.0GHz               11
Intel Core i7 7600U 2.8GHz               11
Intel Pentium Quad Core N4200 1.1GHz     10
Intel Pentium Quad Core N3710 1.6GHz     10
Intel Celeron Dual Core N3050 1.6GHz     10
Intel Core i7 6600U 2.6GHz                9
Intel Core i5 6300U 2.4GHz                8
Intel Core i5 7300U 2.6GHz                7
Intel Core i3 6100U 2.3GHz                7
Intel Core i3 7130U 2.7GHz  

Nos quedamos con la marca, los gigahercios y el tipo de CPU (La gama de Core i, Ryzen o AMD)

In [67]:
df_copy['Cpu_Brand'] = df_copy['Cpu'].str.split().str[0]
df_copy['Cpu_Speed_GHz'] = df_copy['Cpu'].str.extract(r'(\d+\.?\d*)GHz').astype(float)
condiciones = [
    df_copy['Cpu'].str.contains('Core i7'),
    df_copy['Cpu'].str.contains('Core i5'),
    df_copy['Cpu'].str.contains('Core i3'),
    df_copy['Cpu'].str.contains('Ryzen'),
    df_copy['Cpu_Brand'] == 'AMD',]

resultados = [
    'Core i7',
    'Core i5',
    'Core i3',
    'Ryzen',
    'Other AMD',]

df_copy['Cpu_Type'] = np.select(condiciones, resultados, default='Other Intel')
df_copy = df_copy.drop(columns=['Cpu'])
print(df_copy[['Cpu_Brand', 'Cpu_Type', 'Cpu_Speed_GHz']].head(15))

   Cpu_Brand     Cpu_Type  Cpu_Speed_GHz
0      Intel      Core i3            2.0
1      Intel      Core i7            2.6
2      Intel      Core i7            2.7
3      Intel      Core i5            1.8
4      Intel      Core i3            2.0
5      Intel      Core i5            2.6
6      Intel      Core i5            2.5
7      Intel      Core i3            2.4
8      Intel      Core i7            2.8
9      Intel      Core i7            2.8
10     Intel      Core i5            2.9
11     Intel      Core i7            1.8
12     Intel  Other Intel            1.1
13     Intel      Core i7            2.5
14     Intel      Core i7            1.9


Procesamos ahora la columna 'Gpu'

In [68]:
pd.set_option('display.max_rows', None)
print(df_copy['Gpu'].value_counts())
pd.set_option('display.max_rows', 60)

Gpu
Intel HD Graphics 620             185
Intel HD Graphics 520             125
Intel UHD Graphics 620             52
Nvidia GeForce GTX 1050            48
Nvidia GeForce 940MX               31
Nvidia GeForce GTX 1060            31
Intel HD Graphics 400              30
Intel HD Graphics 500              27
Intel HD Graphics                  25
Nvidia GeForce GTX 1070            22
AMD Radeon 530                     22
Nvidia GeForce GTX 1050 Ti         21
AMD Radeon R5 M430                 18
Nvidia GeForce 930MX               17
Nvidia GeForce GTX 960M            12
Intel HD Graphics 515              12
AMD Radeon 520                     11
Intel HD Graphics 615              10
Nvidia GeForce MX150               10
Nvidia GeForce 920MX               10
Intel HD Graphics 505               8
Intel HD Graphics 405               8
AMD Radeon R7 M445                  7
AMD Radeon R5 M420                  7
Nvidia GeForce GTX 950M             7
Intel Iris Plus Graphics 640        6
Nvidia G

Nos quedamos con la marca, la serie y el número de modelo

In [69]:
df_copy['Gpu_Brand'] = df_copy['Gpu'].str.split().str[0]

condiciones2 = [
    df_copy['Gpu'].str.contains('GeForce'),
    df_copy['Gpu'].str.contains('Quadro'),
    df_copy['Gpu'].str.contains('Radeon'),
    df_copy['Gpu'].str.contains('FirePro'),
    df_copy['Gpu'].str.contains('Iris'),
    df_copy['Gpu'].str.contains('UHD Graphics'),
    df_copy['Gpu'].str.contains('HD Graphics'),]

resultados2 = [
    'GeForce', 'Quadro', 'Radeon', 'FirePro', 'Iris', 'UHD Graphics', 'HD Graphics',]

df_copy['Gpu_Series'] = np.select(condiciones2, resultados2, default='Other')

df_copy['Gpu_Model_Number'] = df_copy['Gpu'].str.extract(r'(\d+)(?!.*\d)').astype(float)

df_copy = df_copy.drop(columns=['Gpu'])

print(df_copy[['Gpu_Brand', 'Gpu_Series', 'Gpu_Model_Number']].head(15))

   Gpu_Brand    Gpu_Series  Gpu_Model_Number
0      Intel   HD Graphics             520.0
1     Nvidia       GeForce              39.0
2     Nvidia       GeForce             930.0
3      Intel   HD Graphics            6000.0
4        AMD        Radeon             430.0
5      Intel   HD Graphics             620.0
6      Intel   HD Graphics             620.0
7     Nvidia       GeForce             940.0
8     Nvidia       GeForce            1050.0
9     Nvidia       GeForce             940.0
10     Intel          Iris             550.0
11     Intel   HD Graphics             620.0
12     Intel   HD Graphics             505.0
13     Intel   HD Graphics             520.0
14     Intel  UHD Graphics             620.0


Procesamos la columna 'Memory' y pasamos todos los valores string a numéricos con get_dummies

In [70]:
partes = df_copy['Memory'].str.extractall(r'(\d+\.?\d*)(TB|GB)')
partes.columns = ['Size', 'Unit']
partes['Size'] = partes['Size'].astype(float)
partes.loc[partes['Unit'] == 'TB', 'Size'] = partes.loc[partes['Unit'] == 'TB', 'Size'] * 1000
df_copy['Storage_Total_GB'] = partes.groupby(level=0)['Size'].sum()
df_copy = df_copy.drop(columns=['Memory'])

In [71]:
df_copy = df_copy.drop(columns=['Product'])

In [72]:
categorical_cols = ['Company', 'TypeName', 'OpSys', 'Cpu_Brand', 'Cpu_Type', 'Gpu_Brand', 'Gpu_Series']
numeric_cols = [c for c in df_copy.columns if c not in categorical_cols + ['Price_in_euros']]

Eliminamos ahora las columnas con identificadores únicos, ya que no aportan nada al modelo

### 2.2 Definir X e y


In [73]:
df_copy.rename(columns={'Price_in_euros': 'target'}, inplace=True)

In [74]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 912 entries, 0 to 911
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   laptop_ID         912 non-null    int64  
 1   Company           912 non-null    str    
 2   TypeName          912 non-null    str    
 3   Inches            912 non-null    float64
 4   Ram               912 non-null    int64  
 5   OpSys             912 non-null    str    
 6   Weight            912 non-null    float64
 7   target            912 non-null    float64
 8   Touchscreen       912 non-null    int64  
 9   IPS_Panel         912 non-null    int64  
 10  Retina_Display    912 non-null    int64  
 11  Res_Width         912 non-null    int64  
 12  Res_Height        912 non-null    int64  
 13  Cpu_Brand         912 non-null    object 
 14  Cpu_Speed_GHz     912 non-null    float64
 15  Cpu_Type          912 non-null    str    
 16  Gpu_Brand         912 non-null    object 
 17  Gpu_Seri

In [75]:
X_train, X_test, y_train, y_test = train_test_split(df_copy.drop(columns=['target']), df_copy['target'], test_size=0.2, random_state=42)

In [76]:
X_train.head()

,laptop_ID,Company,TypeName,Inches,Ram,OpSys,Weight,Touchscreen,IPS_Panel,Retina_Display,Res_Width,Res_Height,Cpu_Brand,Cpu_Speed_GHz,Cpu_Type,Gpu_Brand,Gpu_Series,Gpu_Model_Number,Storage_Total_GB
25,1118,HP,Workstation,17.3,8,Windows 7,3.00,0,1,0,1920,1080,Intel,2.6,Core i7,AMD,FirePro,6150.0,1000.0
84,153,Dell,Gaming,15.6,16,Windows 10,2.56,0,0,0,1920,1080,Intel,2.8,Core i7,Nvidia,GeForce,1050.0,512.0
10,275,Apple,Ultrabook,13.3,8,macOS,1.37,0,1,1,2560,1600,Intel,2.9,Core i5,Intel,Iris,550.0,512.0
342,1100,HP,Notebook,14.0,4,Windows 7,1.54,0,0,0,1920,1080,Intel,2.3,Core i5,Intel,HD Graphics,520.0,500.0
890,131,Dell,Notebook,17.3,16,Windows 10,2.80,0,0,0,1920,1080,Intel,1.8,Core i7,AMD,Radeon,530.0,2256.0


In [77]:
# Definir el codificador
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
    ('num', 'passthrough', numeric_cols),
])

# Aprender las categorías SOLO con X_train
preprocessor.fit(X_train)

# Aplicar la transformación a train y a test
X_train_enc = preprocessor.transform(X_train)
X_test_enc = preprocessor.transform(X_test)

c:\Users\sandy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


## 4. Modelado

### 4.1 Entrenamiento

In [79]:
# Tu código aquí
rnd_forest = RandomForestRegressor(max_depth=5, n_estimators=100, random_state=42)
rnd_forest.fit(X_train_enc, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [80]:
# Tu código aquí
predictions = rnd_forest.predict(X_test_enc)
rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")

RMSE: 357.5149700513043


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
# Tu código aquí


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [81]:
# Tu código aquí
x_train2 = pd.read_csv('./data/train.csv', encoding='latin-1')

In [82]:
x_train2['Ram'] = x_train2['Ram'].str.replace('GB', '').astype(int)
x_train2['Weight'] = x_train2['Weight'].str.replace('kg', '').astype(float)

In [83]:
x_train2['Touchscreen'] = x_train2['ScreenResolution'].str.contains('Touchscreen').astype(int)
x_train2['IPS_Panel'] = x_train2['ScreenResolution'].str.contains('IPS Panel').astype(int)
x_train2['Retina_Display'] = x_train2['ScreenResolution'].str.contains('Retina Display').astype(int)

In [84]:
resolucion = x_train2['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
x_train2['Res_Width'] = resolucion[0].astype(int)
x_train2['Res_Height'] = resolucion[1].astype(int)

x_train2 = x_train2.drop(columns=['ScreenResolution'])

In [85]:
x_train2['Cpu_Brand'] = x_train2['Cpu'].str.split().str[0]
x_train2['Cpu_Speed_GHz'] = x_train2['Cpu'].str.extract(r'(\d+\.?\d*)GHz').astype(float)

condiciones_cpu = [
    x_train2['Cpu'].str.contains('Core i7'),
    x_train2['Cpu'].str.contains('Core i5'),
    x_train2['Cpu'].str.contains('Core i3'),
    x_train2['Cpu'].str.contains('Ryzen'),
    x_train2['Cpu_Brand'] == 'AMD',
]
resultados_cpu = ['Core i7', 'Core i5', 'Core i3', 'Ryzen', 'Other AMD']
x_train2['Cpu_Type'] = np.select(condiciones_cpu, resultados_cpu, default='Other Intel')

x_train2 = x_train2.drop(columns=['Cpu'])

In [86]:
x_train2['Gpu_Brand'] = x_train2['Gpu'].str.split().str[0]

condiciones_gpu = [
    x_train2['Gpu'].str.contains('GeForce'),
    x_train2['Gpu'].str.contains('Quadro'),
    x_train2['Gpu'].str.contains('Radeon'),
    x_train2['Gpu'].str.contains('FirePro'),
    x_train2['Gpu'].str.contains('Iris'),
    x_train2['Gpu'].str.contains('UHD Graphics'),
    x_train2['Gpu'].str.contains('HD Graphics'),
]
resultados_gpu = ['GeForce', 'Quadro', 'Radeon', 'FirePro', 'Iris', 'UHD Graphics', 'HD Graphics']
x_train2['Gpu_Series'] = np.select(condiciones_gpu, resultados_gpu, default='Other')

In [87]:
x_train2['Gpu_Model_Number'] = x_train2['Gpu'].str.extract(r'(\d+)(?!.*\d)').astype(float)

x_train2 = x_train2.drop(columns=['Gpu'])

In [88]:
partes = x_train2['Memory'].str.extractall(r'(\d+\.?\d*)(TB|GB)')
partes.columns = ['Size', 'Unit']
partes['Size'] = partes['Size'].astype(float)
partes.loc[partes['Unit'] == 'TB', 'Size'] = partes.loc[partes['Unit'] == 'TB', 'Size'] * 1000

x_train2['Storage_Total_GB'] = partes.groupby(level=0)['Size'].sum()

x_train2 = x_train2.drop(columns=['Memory'])

In [89]:
x_train2 = x_train2.drop(columns=['Product'])

In [90]:
categorical_cols = ['Company', 'TypeName', 'OpSys', 'Cpu_Brand', 'Cpu_Type', 'Gpu_Brand', 'Gpu_Series']
numeric_cols = [c for c in x_train2.columns if c not in categorical_cols + ['Price_in_euros']]

In [92]:
x_train2.rename(columns={'Price_in_euros': 'target'}, inplace=True)

In [93]:
X = x_train2.drop(columns=['target'])
y = x_train2['target']

In [55]:
X.shape

(912, 54)

In [94]:
X.head()

,laptop_ID,Company,TypeName,Inches,Ram,OpSys,Weight,Touchscreen,IPS_Panel,Retina_Display,Res_Width,Res_Height,Cpu_Brand,Cpu_Speed_GHz,Cpu_Type,Gpu_Brand,Gpu_Series,Gpu_Model_Number,Storage_Total_GB
0,755,HP,Notebook,15.6,8,Windows 10,1.86,0,0,0,1920,1080,Intel,2.0,Core i3,Intel,HD Graphics,520.0,256.0
1,618,Dell,Gaming,15.6,16,Windows 10,2.59,0,0,0,1920,1080,Intel,2.6,Core i7,Nvidia,GeForce,39.0,1000.0
2,909,HP,Notebook,15.6,8,Windows 10,2.04,0,0,0,1920,1080,Intel,2.7,Core i7,Nvidia,GeForce,930.0,1000.0
3,2,Apple,Ultrabook,13.3,8,macOS,1.34,0,0,0,1440,900,Intel,1.8,Core i5,Intel,HD Graphics,6000.0,128.0
4,286,Dell,Notebook,15.6,4,Linux,2.25,0,0,0,1920,1080,Intel,2.0,Core i3,AMD,Radeon,430.0,1000.0


In [116]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
    ('num', 'passthrough', numeric_cols),
])
X_enc = preprocessor.fit_transform(X)

In [97]:
rnd_forest = RandomForestRegressor(max_depth=5, n_estimators=100, random_state=42)
rnd_forest.fit(X_enc, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [104]:
y_pred = rnd_forest.predict(X_enc)
y_pred

array([1000.12628605, 1609.36714253,  960.46530702, 1353.70863644,
        474.54051838,  739.25729814,  900.96044326,  929.51161424,
       1220.04054225, 1973.83657003, 1586.92492844,  724.57119063,
        363.6118674 , 1777.76758697, 1230.09132253,  640.38395857,
       1270.05893768, 1646.33323212, 1207.25603655, 1184.07336567,
        322.9299851 , 1492.97893458, 1156.39115368,  318.77341071,
       1207.25603655, 2036.19927357, 1178.28374118, 1214.49974713,
        550.22419635, 1269.84967382, 1448.82081387,  691.18520721,
        946.98413246,  344.28741331,  996.00068244, 1921.42726465,
        327.32158316, 1133.98058304, 1505.03934359, 1207.25603655,
       1722.55403948,  699.22845167, 1179.12834186, 1593.74836539,
        954.56567968,  681.17940389,  511.83678207, 2418.41445427,
       1688.25596278, 1037.21347077,  711.23518877,  331.87432579,
       5076.89096651,  650.61779442, 1035.94891436, 1183.86014525,
       2212.95279216,  509.80350853,  337.19510273, 1442.89157

In [105]:
ids = X['laptop_ID'].reset_index(drop=True)

In [106]:
x_pred = pd.DataFrame({"laptop_ID": ids, "Price_in_euros": y_pred})
x_pred.head()

,laptop_ID,Price_in_euros
0,755,1000.126286
1,618,1609.367143
2,909,960.465307
3,2,1353.708636
4,286,474.540518


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [108]:
x_pred2 = pd.read_csv('./data/test.csv', encoding='latin-1')
x_pred2.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight
0,209,Lenovo,Legion Y520-15IKBN,Gaming,15.6,Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,512GB SSD,Nvidia GeForce GTX 1060,No OS,2.4kg
1,1281,Acer,Aspire ES1-531,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.6GHz,4GB,500GB HDD,Intel HD Graphics 400,Linux,2.4kg
2,1168,Lenovo,V110-15ISK (i3-6006U/4GB/1TB/No,Notebook,15.6,1366x768,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,Intel HD Graphics 520,No OS,1.9kg
3,1231,Dell,Inspiron 7579,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,Windows 10,2.191kg
4,1020,HP,ProBook 640,Notebook,14.0,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4GB,256GB SSD,Intel HD Graphics 620,Windows 10,1.95kg


## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [109]:
# Tu código aquí
x_pred2['Ram'] = x_pred2['Ram'].str.replace('GB', '').astype(int)
x_pred2['Weight'] = x_pred2['Weight'].str.replace('kg', '').astype(float)
x_pred2['Touchscreen'] = x_pred2['ScreenResolution'].str.contains('Touchscreen').astype(int)
x_pred2['IPS_Panel'] = x_pred2['ScreenResolution'].str.contains('IPS Panel').astype(int)
x_pred2['Retina_Display'] = x_pred2['ScreenResolution'].str.contains('Retina Display').astype(int)
resolucion = x_pred2['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
x_pred2['Res_Width'] = resolucion[0].astype(int)
x_pred2['Res_Height'] = resolucion[1].astype(int)
x_pred2 = x_pred2.drop(columns=['ScreenResolution'])
x_pred2['Cpu_Brand'] = x_pred2['Cpu'].str.split().str[0]
x_pred2['Cpu_Speed_GHz'] = x_pred2['Cpu'].str.extract(r'(\d+\.?\d*)GHz').astype(float)
condiciones_cpu = [
    x_pred2['Cpu'].str.contains('Core i7'),
    x_pred2['Cpu'].str.contains('Core i5'),
    x_pred2['Cpu'].str.contains('Core i3'),
    x_pred2['Cpu'].str.contains('Ryzen'),
    x_pred2['Cpu_Brand'] == 'AMD',
]
resultados_cpu = ['Core i7', 'Core i5', 'Core i3', 'Ryzen', 'Other AMD']
x_pred2['Cpu_Type'] = np.select(condiciones_cpu, resultados_cpu, default='Other Intel')
x_pred2 = x_pred2.drop(columns=['Cpu'])
x_pred2['Gpu_Brand'] = x_pred2['Gpu'].str.split().str[0]
condiciones_gpu = [
    x_pred2['Gpu'].str.contains('GeForce'),
    x_pred2['Gpu'].str.contains('Quadro'),
    x_pred2['Gpu'].str.contains('Radeon'),
    x_pred2['Gpu'].str.contains('FirePro'),
    x_pred2['Gpu'].str.contains('Iris'),
    x_pred2['Gpu'].str.contains('UHD Graphics'),
    x_pred2['Gpu'].str.contains('HD Graphics'),
]
resultados_gpu = ['GeForce', 'Quadro', 'Radeon', 'FirePro', 'Iris', 'UHD Graphics', 'HD Graphics']
x_pred2['Gpu_Series'] = np.select(condiciones_gpu, resultados_gpu, default='Other')
x_pred2['Gpu_Model_Number'] = x_pred2['Gpu'].str.extract(r'(\d+)(?!.*\d)').astype(float)
x_pred2 = x_pred2.drop(columns=['Gpu'])
partes = x_pred2['Memory'].str.extractall(r'(\d+\.?\d*)(TB|GB)')
partes.columns = ['Size', 'Unit']
partes['Size'] = partes['Size'].astype(float)
partes.loc[partes['Unit'] == 'TB', 'Size'] = partes.loc[partes['Unit'] == 'TB', 'Size'] * 1000
x_pred2['Storage_Total_GB'] = partes.groupby(level=0)['Size'].sum()
x_pred2 = x_pred2.drop(columns=['Memory'])

In [110]:
x_pred2 = x_pred2.drop(columns=['Product'])

In [112]:
categorical_cols = ['Company', 'TypeName', 'OpSys', 'Cpu_Brand', 'Cpu_Type', 'Gpu_Brand', 'Gpu_Series']
numeric_cols = [c for c in x_pred2.columns if c not in categorical_cols + ['Price_in_euros']]

In [113]:
x_pred2.rename(columns={'Price_in_euros': 'target'}, inplace=True)

In [117]:
x_pred2_enc = preprocessor.transform(x_pred2)

c:\Users\sandy\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [118]:
pred_submit = rnd_forest.predict(x_pred2_enc)
pred_submit

array([1898.40561873,  328.03758841,  515.40929487, 1238.1707019 ,
       1003.32293128,  447.07516507,  850.33402084,  949.59634191,
       1525.89910446,  333.78252104, 1874.2996249 , 1394.11571396,
        514.94669759, 1699.50879974,  763.31766346,  692.84501279,
       1757.21628558, 1408.17092574, 1680.95082474,  627.40884389,
       1782.21597004,  317.17948239,  701.65211378, 1379.58572633,
        548.79713469,  946.98413246,  660.93699054,  506.56711494,
       2861.60426701, 1034.73870617, 2472.99823061,  511.83678207,
        746.08756332, 3129.95174993, 1725.29262637, 1670.1983597 ,
        682.7750802 , 1556.95122046,  952.38116677, 1447.61199243,
        681.7963893 , 1356.54459017,  518.28438049, 1043.07382382,
       1515.09703478, 1178.28374118,  978.57601005,  562.9471529 ,
        947.29954139,  363.6118674 , 1686.75254415,  947.45112837,
       1214.49974713,  682.22626186, 1630.90168752, 1582.05860005,
        627.57728237,  947.45112837, 1191.21932867,  559.88622

In [120]:
ids2 = sample["laptop_ID"]

first_submit = pd.DataFrame({"laptop_ID": ids2, "Price_in_euros": pred_submit})
first_submit.head()


,laptop_ID,Price_in_euros
0,209,1898.405619
1,1281,328.037588
2,1168,515.409295
3,1231,1238.170702
4,1020,1003.322931


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [119]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

,laptop_ID,Price_in_euros
0,209,1949.1
1,1281,805.0
2,1168,1101.0
3,1231,1293.8
4,1020,1832.6


### 8.2 Crea tu submission

In [ ]:
# Tu código aquí


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [121]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [122]:
checker(first_submit, sample)

 ¡Todo correcto! Submission guardada como 'submission_20260703_113224.csv'. ¡A Kaggle!
